# Tutorial: Using kb-mcp with hep-multiagent

This tutorial shows how to give an AI agent access to a searchable knowledge base of physics papers.

In [ ]:
# Install dependencies
%pip install -q git+https://github.com/HEP-KE/kb-mcp.git
%pip install -q git+https://github.com/HEP-KE/HEP-multiagent.git

---
## Step 1: Configuration

Set up paths and choose which papers to download.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

# Paths (stores data in current directory)
DATA_DIR = Path.cwd() / "data"
DB_PATH = DATA_DIR / "kb.db"
PAPERS_DIR = DATA_DIR / "papers"

# Papers to download (arXiv ID, title)
PAPERS = [
    ("1807.06209", "Planck 2018 cosmological parameters"),
    ("2007.08991", "eBOSS cosmological results"),
    ("1502.01589", "Planck 2015 cosmological results"),
]

print(f"Database: {DB_PATH}")
print(f"Papers: {PAPERS_DIR}")

---
## Step 2: Create the database

Initialize an empty SQLite database with the kb-mcp schema.

In [ ]:
from kb_mcp.kb.db_models import Base
from sqlalchemy import create_engine

# Create directories
DB_PATH.parent.mkdir(parents=True, exist_ok=True)
PAPERS_DIR.mkdir(parents=True, exist_ok=True)

# Create database
engine = create_engine(f"sqlite:///{DB_PATH}")
Base.metadata.create_all(engine)

print(f"Created database: {DB_PATH}")

---
## Step 3: Download papers from arXiv

Fetch PDFs from arXiv and extract text.

In [ ]:
from hep_multiagent.features.arxiv_fetch import download_full_text

for arxiv_id, title in PAPERS:
    txt_path = PAPERS_DIR / f"{arxiv_id}.txt"
    if txt_path.exists():
        print(f"[skip] {arxiv_id} - already downloaded")
    else:
        print(f"[downloading] {arxiv_id}: {title}")
        download_full_text(arxiv_id, str(PAPERS_DIR))

print(f"\nDownloaded {len(list(PAPERS_DIR.glob('*.txt')))} papers")

---
## Step 4: Add papers to the knowledge base

Ingest the downloaded papers into kb-mcp.

In [ ]:
os.environ["SQLITE_DB_PATH"] = str(DB_PATH)

for arxiv_id, _ in PAPERS:
    txt_path = PAPERS_DIR / f"{arxiv_id}.txt"
    if txt_path.exists():
        print(f"[ingesting] {arxiv_id}")
        # NOTE: Embeddings required for kb_search to work with SQLite.
        # Using --no-embed causes kb_search to crash (KeyError: 'total_results')
        # because SQLite doesn't support full-text search.
        subprocess.run(
            [sys.executable, "-m", "kb_mcp.kb.cli", "ingest", str(txt_path),
             "--source-id", "arxiv", "--no-summary", "--batch"],
            capture_output=True
        )

print("\nDone! Checking database...")
result = subprocess.run([sys.executable, "-m", "kb_mcp.kb.cli", "stats"], capture_output=True, text=True)
print(result.stdout)

---
## Step 5: Set up the LLM

Load API credentials and initialize the language model.

In [ ]:
from dotenv import load_dotenv
load_dotenv()

from langchain_openai import ChatOpenAI
llm = ChatOpenAI(
    model="claudesonnet4",
    base_url="https://apps-dev.inside.anl.gov/argoapi/v1",
    api_key=os.environ["ARGO_USER"]
)
print("Using Argo API")

---
## Step 6: Create the agent

Initialize hep-multiagent with kb-mcp. The agent gets these tools:
- `kb_search` - search papers by keyword
- `kb_get` - get full text of a paper

In [ ]:
from hep_multiagent import Agent

agent = await Agent(
    llm=llm,
    mcp_servers=[{
        "url": "https://github.com/HEP-KE/kb-mcp.git",
        "name": "kb-server-stdio",
        "env": {"SQLITE_DB_PATH": str(DB_PATH)},
    }],
    approval=False,
)

print(f"Agent ready with {len(agent.tools)} tools:")
for tool in agent.tools:
    print(f"  - {tool.name}")

---
## Step 7: Query the knowledge base

Ask the agent questions. It will use `kb_search` to find relevant papers.

In [ ]:
result = await agent.run(
    query="Search the knowledge base for cosmological parameters. What are the key findings?",
    output_dir="./output"
)

print(result.get("final_report", "No report")[:2000])

In [ ]:
result = await agent.run(
    query="What do the papers say about the Hubble constant and dark energy?",
    output_dir="./output"
)

print(result.get("final_report", "No report")[:2000])

---
## Notes for kb-mcp developers

**Bug: kb_search crashes with SQLite + no embeddings**

When using SQLite (not PostgreSQL) and ingesting without embeddings (`--no-embed`), `kb_search` throws `KeyError: 'total_results'` because:
- Semantic search fails: "Embedding config not found"
- Full-text search fails: "only supported on PostgreSQL"
- The code doesn't handle both searches failing

**Workaround:** Either use PostgreSQL, or ingest with embeddings (slower).

---

**Feature request: On-the-fly INSPIRE search**

kb-mcp has INSPIRE import in the CLI (`kb_mcp.imports.cli inspire --query "..."`), but this is not exposed as an MCP tool. The agent can only search what's already in the database.

Expose INSPIRE search as an MCP tool (e.g., `kb_inspire_search`) so agents can:
1. Search INSPIRE directly without pre-populating a database
2. Optionally auto-ingest results for future queries

This would enable a "zero-setup" workflow where the agent fetches papers on-the-fly.